# OneStopEnglish OOD End-to-End

This notebook combines the older OneStopEnglish OOD notebooks into one workflow:

1. Download and prepare `onestop_english` from Hugging Face.
2. Compute Rooein-style static readability features.
3. Run LLM prompt-metric inference for the 63 prompt questions.
4. Evaluate the frozen ScienceQA-trained Step 7 artifacts on OneStopEnglish.

The separate `onestopenglish_train_InDomain.ipynb` notebook is intentionally kept separate because it trains and tests directly on OneStopEnglish, which is not frozen OOD transfer.


In [ ]:
%pip install -q datasets pandas numpy nltk==3.8.1 textstat==0.7.3 spacy==3.7.4 tqdm scikit-learn joblib torch transformers accelerate bitsandbytes huggingface_hub
!python -m spacy download en_core_web_sm -q


In [ ]:
import os

MY_DRIVE_SUBDIR = 'BeyondFK'
MODEL_TAG = 'mistral-7b'
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
TEXT_COL = 'full_text'
LABEL_COL = 'education_level'
BATCH_SIZE = 4
SAVE_EVERY = 10
MAX_NEW_TOKENS = 8

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = os.path.join('/content/drive/MyDrive', MY_DRIVE_SUBDIR)
except Exception:
    print('Not Colab - using current directory.')
    BASE_DIR = '.'

OOD_DIR = os.path.join(BASE_DIR, 'ood')
PROMPT_DIR = os.path.join(BASE_DIR, 'outputs', 'prompt_metrics')
RESULTS_ROOT = os.path.join(BASE_DIR, 'results')
ARTIFACTS_DIR = os.path.join(RESULTS_ROOT, 'by_prompt_model', MODEL_TAG, 'artifacts')
OOD_RESULTS_DIR = os.path.join(OOD_DIR, 'results')
os.makedirs(OOD_DIR, exist_ok=True)
os.makedirs(OOD_RESULTS_DIR, exist_ok=True)

PREPARED_CSV = os.path.join(OOD_DIR, 'onestopenglish_prepared.csv')
STATIC_CSV = os.path.join(OOD_DIR, 'onestopenglish_with_static.csv')
FEATURES_CSV = os.path.join(OOD_DIR, f'onestopenglish_features_{MODEL_TAG}.csv')
PROGRESS_CSV = os.path.join(OOD_DIR, f'{MODEL_TAG}_ood_prompt_progress.csv')
PROMPT_JSON = os.path.join(PROMPT_DIR, 'prompt_questions.json')

print('BASE_DIR     :', os.path.abspath(BASE_DIR))
print('OOD_DIR      :', os.path.abspath(OOD_DIR))
print('PROMPT_JSON  :', os.path.abspath(PROMPT_JSON))
print('ARTIFACTS_DIR:', os.path.abspath(ARTIFACTS_DIR))
print('PREPARED_CSV :', os.path.abspath(PREPARED_CSV))
print('STATIC_CSV   :', os.path.abspath(STATIC_CSV))
print('FEATURES_CSV :', os.path.abspath(FEATURES_CSV))


In [ ]:
from datasets import load_dataset
import pandas as pd

HF_DATASET = 'onestop_english'
HF_SPLIT = 'train'

ds = load_dataset(HF_DATASET, split=HF_SPLIT)
df_raw = ds.to_pandas()
print('Columns:', df_raw.columns.tolist())
print('Rows:', len(df_raw))
df_raw.head(3)


In [ ]:
def map_labels_onestop_int(series):
    m = {0: 'elementary', 1: 'middle', 2: 'high', '0': 'elementary', '1': 'middle', '2': 'high'}
    out = []
    for v in series:
        if pd.isna(v):
            out.append(None)
            continue
        if isinstance(v, str) and v.strip().isdigit():
            v = int(v.strip())
        out.append(m.get(v))
    return pd.Series(out, index=series.index)

def build_prepared_df(df, text_col='text', label_col='label'):
    out = df.copy()
    out['full_text'] = out[text_col].astype(str)
    out['education_level'] = map_labels_onestop_int(out[label_col])
    for c in ('text_question', 'text_solution', 'text_lecture'):
        if c not in out.columns:
            out[c] = ''
        else:
            out[c] = out[c].fillna('').astype(str)
    out = out.dropna(subset=['education_level'])
    out = out[out['full_text'].str.strip().astype(bool)]
    return out

df_prep = build_prepared_df(df_raw)
print('Prepared rows:', len(df_prep))
print(df_prep['education_level'].value_counts())


In [ ]:
df_prep.to_csv(PREPARED_CSV, index=False)
print('Saved prepared OneStopEnglish CSV:', PREPARED_CSV)
df_prep.head(2)


## Compute Static Metrics

This section replaces `OOD_Run_Static.ipynb`. It computes the same Rooein-style static columns expected by the Step 7 artifact schema.

In [ ]:
import re
import numpy as np
import nltk
import spacy
import textstat
from tqdm import tqdm
from nltk import pos_tag
from nltk.corpus import cmudict, stopwords, wordnet
from nltk.tokenize import sent_tokenize, word_tokenize

for resource in [
    'stopwords', 'cmudict', 'wordnet', 'averaged_perceptron_tagger',
    'punkt', 'punkt_tab', 'averaged_perceptron_tagger_eng'
]:
    nltk.download(resource, quiet=True)

nlp = spacy.load('en_core_web_sm')
STOP_WORDS = set(stopwords.words('english'))
CMU_DICT = cmudict.dict()
AUXILIARY_VERBS = {
    'be', 'am', 'is', 'are', 'was', 'were', 'been', 'being',
    'have', 'has', 'had', 'having', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must',
    'can', 'could'
}

STATIC_METRIC_NAMES = [
    'n_words_q', 'n_words_a_solution', 'n_words_a_lecture',
    'Text_Length', 'Word_Count', 'Nouns', 'Verbs', 'Adjectives', 'Adverbs',
    'Num_Numbers', 'Num_Commas', 'Num_Complex_Words', 'Num_Unique_Words',
    'Num_Content_Words', 'Num_Content_Words_No_Stopwords',
    'Word_Length_Syllables', 'Avg_Sentence_Length',
    'Num_Prepositional_Phrases', 'Num_Negated_Words_Stem',
    'Num_Negated_Words_Lead_In', 'Num_Main_Noun_Phrases',
    'Avg_Main_NP_Length', 'Num_Verb_Phrases', 'Prop_Active_Voice_Verbs',
    'Prop_Passive_Voice_Verbs', 'Ratio_Active_to_Passive_Verbs',
    'Num_Words_Before_Main_Verb', 'Num_Agentless_Passive_Constructions',
    'Word_Length_Std_Dev', 'Num_Polysemic_Words', 'Num_Word_Senses',
    'Num_Word_Senses_For_Content_Words', 'Num_Word_Senses_For_Nouns',
    'Num_Word_Senses_For_Verbs', 'Num_Word_Senses_For_Non_Auxiliary_Verbs',
    'Num_Word_Senses_For_Adjectives', 'Num_Word_Senses_For_Adverbs',
    'Distance_To_Root_Nouns', 'Distance_To_Root_Verbs',
    'flesch_kincaid_grade', 'flesch_kincaid_ease', 'coleman_liau_index',
    'automated_readability_index', 'smog_index', 'gunning_fog',
    'traenkle_bailer_index'
]


def count_syllables(word):
    w = str(word).lower()
    if w in CMU_DICT:
        return len([ph for ph in CMU_DICT[w][0] if ph[-1].isdigit()])
    w = re.sub(r'[^a-z]', '', w)
    if not w:
        return 0
    count = 0
    vowels = 'aeiouy'
    prev = False
    for ch in w:
        is_vowel = ch in vowels
        if is_vowel and not prev:
            count += 1
        prev = is_vowel
    if w.endswith('e') and count > 1:
        count -= 1
    return max(count, 1)


def is_complex_word(word):
    return count_syllables(word) >= 3


def count_word_senses(word, pos=None):
    return len(wordnet.synsets(str(word), pos=pos))


def avg_distance_to_root(word, pos):
    synsets = wordnet.synsets(str(word), pos=pos)
    if not synsets:
        return 0.0
    return float(np.mean([s.min_depth() for s in synsets]))


def _zero_static_metrics():
    return {m: 0 for m in STATIC_METRIC_NAMES}


def compute_static_metrics(row):
    text = str(row.get('full_text', ''))
    question = str(row.get('text_question', ''))
    solution = str(row.get('text_solution', ''))
    lecture = str(row.get('text_lecture', ''))
    if not text.strip():
        return _zero_static_metrics()

    try:
        words = word_tokenize(text)
        words_alpha = [w for w in words if w.isalpha()]
        words_lower = [w.lower() for w in words_alpha]
        sentences = sent_tokenize(text)
        tagged = pos_tag(words_alpha)
        nouns = [w for w, t in tagged if t.startswith('NN')]
        verbs = [w for w, t in tagged if t.startswith('VB')]
        adjectives = [w for w, t in tagged if t.startswith('JJ')]
        adverbs = [w for w, t in tagged if t.startswith('RB')]
        doc = nlp(text[:100000])

        word_count = len(words_alpha)
        n_words_q = len(word_tokenize(question))
        n_words_a_solution = len(word_tokenize(solution))
        n_words_a_lecture = len(word_tokenize(lecture))
        text_length = len(text)
        num_numbers = sum(1 for c in text if c.isdigit())
        num_commas = text.count(',')
        num_complex = sum(1 for w in words_alpha if is_complex_word(w))
        num_unique = len(set(words_lower))

        content_pos_words = set(w.lower() for w in nouns + verbs + adjectives + adverbs)
        num_content = len([w for w in words_lower if w in content_pos_words])
        content_words_no_stop = [w for w in words_lower if w not in STOP_WORDS]
        num_content_no_stop = len(content_words_no_stop)
        syllables = [count_syllables(w) for w in words_alpha]
        avg_syllables = float(np.mean(syllables)) if syllables else 0.0
        sent_lengths = [len(word_tokenize(s)) for s in sentences]
        avg_sent_len = float(np.mean(sent_lengths)) if sent_lengths else 0.0

        num_prep_phrases = sum(1 for token in doc if token.pos_ == 'ADP')
        num_negated_stem = sum(1 for token in doc if token.dep_ == 'neg')
        neg_words = {'no', 'not', 'never', 'neither', 'nobody', 'nothing', 'nowhere', 'nor'}
        num_negated_lead_in = sum(1 for w in words_lower if w in neg_words)
        noun_chunks = list(doc.noun_chunks)
        num_main_np = len(noun_chunks)
        np_lengths = [len(chunk) for chunk in noun_chunks]
        avg_np_length = float(np.mean(np_lengths)) if np_lengths else 0.0
        num_verb_phrases = len([token for token in doc if token.pos_ == 'VERB'])

        passive_verbs, active_verbs = set(), set()
        for token in doc:
            if token.dep_ == 'nsubjpass':
                passive_verbs.add(token.head.i)
            if token.dep_ == 'nsubj' and token.head.pos_ == 'VERB':
                active_verbs.add(token.head.i)
        passive_count = len(passive_verbs)
        active_count = len(active_verbs)
        total_voice = active_count + passive_count
        prop_active = active_count / total_voice if total_voice > 0 else 1.0
        prop_passive = passive_count / total_voice if total_voice > 0 else 0.0
        ratio_active_passive = active_count / passive_count if passive_count > 0 else float(active_count)

        agentless_passive = 0
        for token in doc:
            if token.dep_ == 'nsubjpass':
                has_agent = any(child.dep_ == 'agent' for child in token.head.children)
                if not has_agent:
                    agentless_passive += 1

        words_before_verb_list = []
        for sent in doc.sents:
            for i, token in enumerate(sent):
                if token.pos_ == 'VERB' and token.dep_ in ('ROOT', 'ccomp', 'advcl'):
                    words_before_verb_list.append(i)
                    break
        words_before_verb = float(np.mean(words_before_verb_list)) if words_before_verb_list else 0.0

        word_len_std = float(np.std([len(w) for w in words_alpha])) if words_alpha else 0.0
        unique_words_lower = set(words_lower)
        num_polysemic = sum(1 for w in unique_words_lower if count_word_senses(w) > 1)
        total_senses = sum(count_word_senses(w) for w in unique_words_lower)
        unique_content_no_stop = set(content_words_no_stop)
        content_word_senses = sum(count_word_senses(w) for w in unique_content_no_stop)
        noun_senses = sum(count_word_senses(w.lower(), wordnet.NOUN) for w in set(nouns))
        verb_senses = sum(count_word_senses(w.lower(), wordnet.VERB) for w in set(verbs))
        non_aux_verbs = [w for w in verbs if w.lower() not in AUXILIARY_VERBS]
        non_aux_verb_senses = sum(count_word_senses(w.lower(), wordnet.VERB) for w in set(non_aux_verbs))
        adj_senses = sum(count_word_senses(w.lower(), wordnet.ADJ) for w in set(adjectives))
        adv_senses = sum(count_word_senses(w.lower(), wordnet.ADV) for w in set(adverbs))
        noun_depths = [avg_distance_to_root(w.lower(), wordnet.NOUN) for w in set(nouns)]
        verb_depths = [avg_distance_to_root(w.lower(), wordnet.VERB) for w in set(verbs)]
        dist_root_nouns = float(np.mean(noun_depths)) if noun_depths else 0.0
        dist_root_verbs = float(np.mean(verb_depths)) if verb_depths else 0.0

        fk_grade = textstat.flesch_kincaid_grade(text)
        fk_ease = textstat.flesch_reading_ease(text)
        cl_index = textstat.coleman_liau_index(text)
        ari = textstat.automated_readability_index(text)
        smog = textstat.smog_index(text)
        gunning = textstat.gunning_fog(text)
        prop_prep = num_prep_phrases / word_count if word_count > 0 else 0.0
        traenkle_bailer = 224.6814 - (79.8304 * (avg_sent_len / 100.0)) - (12.24032 * (prop_prep * 100.0)) if word_count > 0 else 0.0

        return {
            'n_words_q': n_words_q, 'n_words_a_solution': n_words_a_solution, 'n_words_a_lecture': n_words_a_lecture,
            'Text_Length': text_length, 'Word_Count': word_count,
            'Nouns': len(nouns), 'Verbs': len(verbs), 'Adjectives': len(adjectives), 'Adverbs': len(adverbs),
            'Num_Numbers': num_numbers, 'Num_Commas': num_commas,
            'Num_Complex_Words': num_complex, 'Num_Unique_Words': num_unique,
            'Num_Content_Words': num_content, 'Num_Content_Words_No_Stopwords': num_content_no_stop,
            'Word_Length_Syllables': avg_syllables, 'Avg_Sentence_Length': avg_sent_len,
            'Num_Prepositional_Phrases': num_prep_phrases, 'Num_Negated_Words_Stem': num_negated_stem,
            'Num_Negated_Words_Lead_In': num_negated_lead_in, 'Num_Main_Noun_Phrases': num_main_np,
            'Avg_Main_NP_Length': avg_np_length, 'Num_Verb_Phrases': num_verb_phrases,
            'Prop_Active_Voice_Verbs': prop_active, 'Prop_Passive_Voice_Verbs': prop_passive,
            'Ratio_Active_to_Passive_Verbs': ratio_active_passive, 'Num_Words_Before_Main_Verb': words_before_verb,
            'Num_Agentless_Passive_Constructions': agentless_passive, 'Word_Length_Std_Dev': word_len_std,
            'Num_Polysemic_Words': num_polysemic, 'Num_Word_Senses': total_senses,
            'Num_Word_Senses_For_Content_Words': content_word_senses, 'Num_Word_Senses_For_Nouns': noun_senses,
            'Num_Word_Senses_For_Verbs': verb_senses, 'Num_Word_Senses_For_Non_Auxiliary_Verbs': non_aux_verb_senses,
            'Num_Word_Senses_For_Adjectives': adj_senses, 'Num_Word_Senses_For_Adverbs': adv_senses,
            'Distance_To_Root_Nouns': dist_root_nouns, 'Distance_To_Root_Verbs': dist_root_verbs,
            'flesch_kincaid_grade': fk_grade, 'flesch_kincaid_ease': fk_ease,
            'coleman_liau_index': cl_index, 'automated_readability_index': ari,
            'smog_index': smog, 'gunning_fog': gunning, 'traenkle_bailer_index': traenkle_bailer,
        }
    except Exception as e:
        return _zero_static_metrics()

print('Static metric count:', len(STATIC_METRIC_NAMES))


In [ ]:
df_static_input = pd.read_csv(PREPARED_CSV)
if 'full_text' not in df_static_input.columns:
    if 'text' not in df_static_input.columns:
        raise ValueError('Need full_text or text column in prepared CSV')
    df_static_input['full_text'] = df_static_input['text'].astype(str)

for c in ('text_question', 'text_solution', 'text_lecture'):
    if c not in df_static_input.columns:
        df_static_input[c] = ''
    else:
        df_static_input[c] = df_static_input[c].fillna('').astype(str)

rows = []
for i in tqdm(range(len(df_static_input)), desc='Computing static metrics'):
    rows.append(compute_static_metrics(df_static_input.iloc[i]))

metrics_df = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)
for c in STATIC_METRIC_NAMES:
    if c not in metrics_df.columns:
        metrics_df[c] = 0.0
metrics_df = metrics_df[STATIC_METRIC_NAMES]

static_out = pd.concat([df_static_input.reset_index(drop=True), metrics_df], axis=1)
static_out.to_csv(STATIC_CSV, index=False)
print('Saved static-feature CSV:', STATIC_CSV)
print('Shape:', static_out.shape)
static_out.head(2)


## Run Prompt Metrics

This section replaces `OOD_Run_Prompt.ipynb`. It runs the Step 4 prompt questions on the OneStopEnglish OOD texts and appends prompt feature columns.

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def parse_yes_no(response: str):
    r = (response or '').strip().lower()
    if r.startswith('yes'):
        return 1
    if r.startswith('no'):
        return 0
    return None


def extract_prompts(prompt_json_obj):
    prompts = []
    if isinstance(prompt_json_obj, list):
        source = prompt_json_obj
    elif isinstance(prompt_json_obj, dict):
        source = []
        for _, v in prompt_json_obj.items():
            if isinstance(v, list):
                source.extend(v)
    else:
        raise ValueError('Unsupported prompt_questions.json format')

    for i, p in enumerate(source):
        if isinstance(p, dict):
            raw_id = p.get('id', i)
            q = p.get('question', p.get('prompt', str(p)))
        else:
            raw_id = i
            q = str(p)
        pid = f'prompt_{int(raw_id)}' if str(raw_id).isdigit() else f'prompt_{i}'
        prompts.append((pid, q))
    return prompts


def build_prompt_input(text, question):
    return (
        'You are a readability evaluator.\n'
        'Answer with one word only: yes or no.\n\n'
        f'Text:\n{text}\n\n'
        f'Question: {question}\n'
        'Answer:'
    )

assert os.path.exists(STATIC_CSV), f'Missing static CSV: {STATIC_CSV}'
assert os.path.exists(PROMPT_JSON), f'Missing prompt JSON: {PROMPT_JSON}'

df_prompt_input = pd.read_csv(STATIC_CSV)
with open(PROMPT_JSON, 'r', encoding='utf-8') as f:
    prompt_obj = json.load(f)
prompts = extract_prompts(prompt_obj)

print('Rows:', len(df_prompt_input))
print('Prompts:', len(prompts))
print('Loading model:', MODEL_ID)

bnb = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
model.eval()
print('Prompt model loaded.')


In [ ]:
@torch.no_grad()
def batched_yes_no(model_inputs, max_new_tokens=8):
    enc = tokenizer(
        model_inputs,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = [parse_yes_no(x) for x in decoded]
    return parsed, decoded

if os.path.exists(PROGRESS_CSV):
    done_df = pd.read_csv(PROGRESS_CSV)
    start_idx = len(done_df)
    results = done_df.to_dict(orient='records')
    print(f'Resuming from row {start_idx}')
else:
    start_idx = 0
    results = []
    print('Starting fresh')

for i in tqdm(range(start_idx, len(df_prompt_input)), desc=f'{MODEL_TAG} OOD prompt inference'):
    row = df_prompt_input.iloc[i].to_dict()
    text = str(row.get(TEXT_COL, ''))
    feature_values = {}

    for j in range(0, len(prompts), BATCH_SIZE):
        prompt_batch = prompts[j:j + BATCH_SIZE]
        model_inputs = [build_prompt_input(text, q) for _, q in prompt_batch]
        parsed, _ = batched_yes_no(model_inputs, max_new_tokens=MAX_NEW_TOKENS)

        for (pid, q), val in zip(prompt_batch, parsed):
            if val is None:
                retry_input = f'Answer ONLY yes or no.\n\nText:\n{text}\n\nQuestion: {q}\nAnswer:'
                retry_parsed, _ = batched_yes_no([retry_input], max_new_tokens=MAX_NEW_TOKENS)
                val = retry_parsed[0]
                if val is None:
                    val = 0
            feature_values[pid] = int(val)

    out_row = dict(row)
    out_row.update(feature_values)
    results.append(out_row)

    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(PROGRESS_CSV, index=False)

features_out = pd.DataFrame(results)
features_out.to_csv(PROGRESS_CSV, index=False)
features_out.to_csv(FEATURES_CSV, index=False)
print('Saved progress:', PROGRESS_CSV)
print('Saved final features:', FEATURES_CSV)
print('Shape:', features_out.shape)
features_out.head(2)


## Evaluate Frozen Step 7 Artifacts

This section replaces `OOD_Evaluation_From_Artifacts.ipynb`. It loads the saved ScienceQA-trained Step 7 models and evaluates them on the OneStopEnglish feature CSV. This is frozen OOD transfer, not retraining.

In [ ]:
import joblib
from sklearn.metrics import classification_report, f1_score


def normalize_prompt_headers(df):
    rename_map = {}
    for c in df.columns:
        m = re.match(r'^prompt_(\d+)$', str(c).strip())
        if m:
            rename_map[c] = f'prompt_{int(m.group(1))}'
    return df.rename(columns=rename_map)


def prep_numeric(df, cols):
    X = df[cols].copy()
    X = X.apply(pd.to_numeric, errors='coerce')
    return X.fillna(0).replace([np.inf, -np.inf], 0)


def load_optional_joblib(path):
    return joblib.load(path) if os.path.exists(path) else None

schema_path = os.path.join(ARTIFACTS_DIR, f'feature_schema_{MODEL_TAG}.json')
if not os.path.exists(schema_path):
    schema_path = os.path.join(ARTIFACTS_DIR, 'feature_schema.json')
assert os.path.exists(schema_path), f'Missing feature schema: {schema_path}'

with open(schema_path, 'r', encoding='utf-8') as f:
    schema = json.load(f)

selector_static = joblib.load(os.path.join(ARTIFACTS_DIR, f'selector_static_{MODEL_TAG}.joblib'))
model_static = joblib.load(os.path.join(ARTIFACTS_DIR, f'model_static_{MODEL_TAG}.joblib'))
scaler_static = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'scaler_static_{MODEL_TAG}.joblib'))

selector_prompt = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'selector_prompt_{MODEL_TAG}.joblib'))
model_prompt = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'model_prompt_{MODEL_TAG}.joblib'))
scaler_prompt = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'scaler_prompt_{MODEL_TAG}.joblib'))

selector_combo = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'selector_combo_{MODEL_TAG}.joblib'))
model_combo = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'model_combo_{MODEL_TAG}.joblib'))
scaler_combo = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'scaler_combo_{MODEL_TAG}.joblib'))

label_encoder = load_optional_joblib(os.path.join(ARTIFACTS_DIR, f'label_encoder_{MODEL_TAG}.joblib'))
label_order = list(getattr(label_encoder, 'classes_', ['elementary', 'high', 'middle']))
LABEL_COL = schema.get('label_col', LABEL_COL)

print('Schema:', schema_path)
print('Static columns:', len(schema.get('static_columns', [])))
print('Prompt columns:', len(schema.get('prompt_columns', [])))
print('Combo columns:', len(schema.get('combo_columns', [])))


In [ ]:
assert os.path.exists(FEATURES_CSV), f'Missing feature CSV: {FEATURES_CSV}'
df_eval = pd.read_csv(FEATURES_CSV)
df_eval.columns = df_eval.columns.astype(str).str.strip()
df_eval = normalize_prompt_headers(df_eval)

missing_static = [c for c in schema['static_columns'] + [LABEL_COL] if c not in df_eval.columns]
if missing_static:
    raise ValueError(f'Missing STATIC columns (first 15): {missing_static[:15]}')

y_true = df_eval[LABEL_COL].astype(str)
results = {
    'dataset': 'OneStopEnglish',
    'model_tag': MODEL_TAG,
    'n_samples': int(len(df_eval)),
    'label_col': LABEL_COL,
    'static_macro_f1': None,
    'prompt_macro_f1': None,
    'combo_macro_f1': None,
    'static_report': None,
    'prompt_report': None,
    'combo_report': None,
    'notes': [],
}

# Step 7 artifact order: selector -> scaler -> model.
X_static = prep_numeric(df_eval, schema['static_columns'])
X_static_sel = selector_static.transform(X_static)
if scaler_static is not None:
    X_static_sel = scaler_static.transform(X_static_sel)
pred_static = model_static.predict(X_static_sel).astype(str)
results['static_macro_f1'] = float(f1_score(y_true, pred_static, average='macro'))
results['static_report'] = classification_report(y_true, pred_static, labels=label_order, output_dict=True, zero_division=0)

missing_prompt = [c for c in schema.get('prompt_columns', []) if c not in df_eval.columns]
if selector_prompt is None or model_prompt is None:
    results['notes'].append('PROMPT skipped: prompt artifacts are missing.')
elif scaler_prompt is None:
    results['notes'].append(f'PROMPT skipped: missing scaler_prompt_{MODEL_TAG}.joblib; re-run Step 7 artifact export if needed.')
elif missing_prompt:
    results['notes'].append(f'PROMPT skipped: missing prompt columns: {missing_prompt[:10]}')
else:
    X_prompt = prep_numeric(df_eval, schema['prompt_columns'])
    X_prompt_sel = scaler_prompt.transform(selector_prompt.transform(X_prompt))
    pred_prompt = model_prompt.predict(X_prompt_sel).astype(str)
    results['prompt_macro_f1'] = float(f1_score(y_true, pred_prompt, average='macro'))
    results['prompt_report'] = classification_report(y_true, pred_prompt, labels=label_order, output_dict=True, zero_division=0)

missing_combo = [c for c in schema.get('combo_columns', []) if c not in df_eval.columns]
if selector_combo is None or model_combo is None:
    results['notes'].append('COMBO skipped: combo artifacts are missing.')
elif scaler_combo is None:
    results['notes'].append(f'COMBO skipped: missing scaler_combo_{MODEL_TAG}.joblib; re-run Step 7 artifact export if needed.')
elif missing_combo:
    results['notes'].append(f'COMBO skipped: missing combo columns: {missing_combo[:10]}')
else:
    X_combo = prep_numeric(df_eval, schema['combo_columns'])
    X_combo_sel = scaler_combo.transform(selector_combo.transform(X_combo))
    pred_combo = model_combo.predict(X_combo_sel).astype(str)
    results['combo_macro_f1'] = float(f1_score(y_true, pred_combo, average='macro'))
    results['combo_report'] = classification_report(y_true, pred_combo, labels=label_order, output_dict=True, zero_division=0)

summary_path = os.path.join(OOD_RESULTS_DIR, f'onestopenglish_ood_eval_{MODEL_TAG}.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"STATIC macro-F1: {results['static_macro_f1']:.4f}")
print(f"PROMPT macro-F1: {results['prompt_macro_f1']:.4f}" if results['prompt_macro_f1'] is not None else 'PROMPT macro-F1: skipped')
print(f"COMBO  macro-F1: {results['combo_macro_f1']:.4f}" if results['combo_macro_f1'] is not None else 'COMBO  macro-F1: skipped')
for note in results['notes']:
    print('NOTE:', note)
print('Saved:', summary_path)
